In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.metrics import accuracy_score, classification_report, log_loss

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATASET_PATH = Path("/content/drive/MyDrive/Colab Notebooks/tdcsfog")
TRAIN_PATH = DATASET_PATH / "train"
TEST_PATH = DATASET_PATH / "test"


In [5]:
def load_file(file_path: Path) -> tuple[np.ndarray, float]:

    # Load the data into a numpy ndarray
    with open(file_path, "rb") as infile:
        arr = np.load(infile)

    # Extract the accelerometer data as the input features
    features = arr[:, 1:4]

    # Extract the labels
    labels = arr[:, 4:]
    labels = np.max(labels, axis=-1)
    return features, np.any(labels).astype(float)

In [6]:
train_files = [f for f in TRAIN_PATH.iterdir() if f.is_file() and f.suffix == ".npy"]

train_df = {"file_name": [], "label": []}
X_train = np.zeros(shape=(len(train_files), 3, 1280))
y_train = np.zeros(shape=(len(train_files),))
for idx, train_file in enumerate(tqdm(train_files, desc="Processing files")):
    # Get features and corresponding label
    features, label = load_file(train_file)
    X_train[idx, :, :] = features.T  # note we transpote features array
    y_train[idx] = label

    # Bookkeeping
    train_df["file_name"].append(train_file.name)
    train_df["label"].append(label)
train_df = pd.DataFrame(train_df)
if not DATASET_PATH.joinpath("train.csv").exists():
    train_df.to_csv(DATASET_PATH / "train.csv", index=False, header=True)

Processing files: 100%|██████████| 4136/4136 [01:19<00:00, 52.01it/s] 


In [7]:
test_files = [f for f in TEST_PATH.iterdir() if f.is_file() and f.suffix == ".npy"]

test_df = {"file_name": [], "label": []}
X_test = np.zeros(shape=(len(test_files), 3, 1280))
y_test = np.zeros(shape=(len(test_files),))
for idx, test_file in enumerate(tqdm(test_files, desc="Processing files")):
    # Get features and corresponding label
    features, label = load_file(test_file)
    X_test[idx, :, :] = features.T  # note we transpote features array
    y_test[idx] = label

    # Bookkeeping
    test_df["file_name"].append(test_file.name)
    test_df["label"].append(label)
test_df = pd.DataFrame(test_df)
if not DATASET_PATH.joinpath("test.csv").exists():
    test_df.to_csv(DATASET_PATH / "test.csv", index=False, header=True)

Processing files: 100%|██████████| 960/960 [00:26<00:00, 36.67it/s] 


In [15]:
train_df.head()

,file_name,label
0,76c7edf878_0004.npy,1.0
1,e7b5afe544_0014.npy,1.0
2,76c7edf878_0011.npy,1.0
3,2a9faf5644_0000.npy,0.0
4,5a70242f37_0002.npy,1.0


In [8]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((4136, 3, 1280), (4136,), (960, 3, 1280), (960,))

In [9]:
X_train_seq = X_train.transpose(0, 2, 1)
X_train_seq.shape

X_test_seq = X_test.transpose(0, 2, 1)
X_test_seq.shape

(960, 1280, 3)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(1280, 3), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Or 'softmax' if multi-class
])

model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,521 (76.25 KB)

 Trainable params: 19,521 (76.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history= model.fit(X_train_seq, y_train, validation_data=(X_test_seq, y_test), epochs=10, batch_size=32)


Epoch 1/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 11s 50ms/step - accuracy: 0.5759 - loss: 0.6679 - val_accuracy: 0.5656 - val_loss: 0.6652
Epoch 2/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.6070 - loss: 0.6351 - val_accuracy: 0.5250 - val_loss: 0.6661
Epoch 3/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.6073 - loss: 0.6360 - val_accuracy: 0.5208 - val_loss: 0.6645
Epoch 4/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.6098 - loss: 0.6220 - val_accuracy: 0.5490 - val_loss: 0.6468
Epoch 5/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.5949 - loss: 0.6418 - val_accuracy: 0.6042 - val_loss: 0.6240
Epoch 6/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.6051 - loss: 0.6170 - val_accuracy: 0.6010 - val_loss: 0.6256
Epoch 7/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 0.6170 - loss: 0.6132 - val_accuracy: 0.5677 - val_loss: 0.7567
Epoch 8/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.5841 - loss: 0.6647 - val

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

y_pred_prob = model.predict(X_test_seq).ravel()
# Convert probabilities to class labels
y_pred = (y_pred_prob > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Accuracy: 0.63125
Precision: 0.5484764542936288
Recall: 0.5089974293059126
F1 Score: 0.528
AUC: 0.6877124424295087
Confusion Matrix:
 [[408 163]
 [191 198]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.68      0.71      0.70       571
         1.0       0.55      0.51      0.53       389

    accuracy                           0.63       960
   macro avg       0.61      0.61      0.61       960
weighted avg       0.63      0.63      0.63       960

